# 04 BM25

## Objetivos

* Implementar BM25 y comparar su ranking con el de TF-IDF + coseno sobre la misma colección de documentos.
* Calcular el aporte de cada término en BM25, identificar qué controlan $k_1$ y $b$, y justificar un cambio de posición usando las estadísticas de los documentos.

**1. Carga el corpus de Project Gutenberg**

Utilice el corpus de Project Gutenberg y el preprocesamiento de la práctica anterior.

Recupere una tabla con los campos **Identificador de documento | Título | Texto**

Calcule la columna `tokens` para la tabla, con lista de tokens normalizados de cada documento.

Calcula DF e IDF.

In [ ]:
import os
import re
import urllib.request
import pandas as pd
import numpy as np
from collections import Counter
import math

corpus_dir = "gutenberg_corpus_100"
os.makedirs(corpus_dir, exist_ok=True)

corpus = []

def download_gutenberg_corpus():
    print("Descargando una muestra de libros de Project Gutenberg...")
    count = 0
    book_id = 11
    while count < 10:
        url = f"https://www.gutenberg.org/files/{book_id}/{book_id}-0.txt"
        file_path = os.path.join(corpus_dir, f"book_{book_id}.txt")
        try:
            urllib.request.urlretrieve(url, file_path)
            count += 1
        except Exception:
            pass
        book_id += 1

txt_files = [f for f in os.listdir(corpus_dir) if f.endswith(".txt")]
if len(txt_files) == 0:
    download_gutenberg_corpus()
    txt_files = [f for f in os.listdir(corpus_dir) if f.endswith(".txt")]

for filename in sorted(txt_files):
    filepath = os.path.join(corpus_dir, filename)
    with open(filepath, 'r', encoding='utf-8', errors='ignore') as file_f:
        text = file_f.read()
    title = "Untitled"
    for line in text.split('\n')[:15]:
        if line.startswith("Title:"):
            title = line.replace("Title:", "").strip()
            break
    doc_id = filename.replace("book_", "").replace(".txt", "")
    corpus.append({
        "id": doc_id,
        "title": title,
        "text": text
    })

print(f"Documentos cargados: {len(corpus)}")

data_punto1 = []
corpus_tokens_list = []
for doc in corpus:
    tokens = re.findall(r'\b\w+\b', doc['text'].lower())
    corpus_tokens_list.append(tokens)
    data_punto1.append({
        "Identificador de documento": doc['id'],
        "Título": doc['title'],
        "Texto": doc['text'][:200] + "...",
        "tokens": tokens
    })

df_punto1 = pd.DataFrame(data_punto1, columns=["Identificador de documento", "Título", "Texto", "tokens"])
print("\n--- Primeras 5 filas del DataFrame procesado ---")
print(df_punto1[['Identificador de documento', 'Título', 'tokens']].head().to_string())

def compute_df(corpus_tokens_list):
    df = Counter()
    for doc_tokens in corpus_tokens_list:
        unique_terms = set(doc_tokens)
        for term in unique_terms:
            df[term] += 1
    return df

def compute_idf(df, num_docs):
    idf = {}
    for term, count in df.items():
        idf[term] = math.log10(num_docs / count)
    return idf

num_docs = len(corpus)
df_global = compute_df(corpus_tokens_list)
idf_global = compute_idf(df_global, num_docs)

print(f"\nTamaño del vocabulario: {len(df_global)} términos únicos.")

Documentos cargados: 10

--- Primeras 5 filas del DataFrame procesado ---
  Identificador de documento    Título                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    tokens
0                         11  Untitled                                                                       

**2. Benchmarking**

Seleccione **ocho consultas en el idioma del corpus** y registra en una línea qué busca el usuario con cada una. Incluye términos muy frecuentes, términos poco frecuentes, y consultas de distintas longitudes. Usa $df$ para justificar las elecciones.

Para **TF-IDF + coseno** conserva la representación dispersa de la matriz.

Para **BM25** define $k_1=1.2$, $b=0.75$. Mantén intactas las estadísticas de la colección al cambiar de consulta.

Para cada consulta, ordena por puntuación descendente y desempata por `doc_id` ascendente (trata los IDs como cadenas en ambos métodos). Presenta hasta diez documentos con score positivo por método. Una consulta sin términos en el vocabulario produce dos listas vacías.

In [13]:
queries = [
    {"id": 1, "text": "alice in wonderland", "desc": "Busca la obra de Alicia usando términos comunes en algunos libros pero nulos en otros.", "intent": "Específica (Alicia)"},
    {"id": 2, "text": "moby dick whale", "desc": "Busca el libro Moby Dick usando términos específicos de alta concentración en ese documento.", "intent": "Específica (Moby Dick)"},
    {"id": 3, "text": "the project gutenberg", "desc": "Consulta con términos extremadamente comunes presentes en prácticamente todos los documentos.", "intent": "Genérica (Muy frecuentes)"},
    {"id": 4, "text": "hunting of the snark", "desc": "Consulta de longitud media que coincide de manera muy exacta con el título de un documento específico (ID 13).", "intent": "Específica (Frase exacta)"},
    {"id": 5, "text": "world factbook central intelligence", "desc": "Consulta técnica con términos muy específicos del documento de datos (ID 14).", "intent": "Específica (Términos técnicos)"},
    {"id": 6, "text": "bell captain sea", "desc": "Consulta combinada con términos de frecuencia variada y común en novelas de aventura.", "intent": "Variada (Frecuencia media/baja)"},
    {"id": 7, "text": "serpent dragon", "desc": "Consulta con términos de baja frecuencia (raros) dentro del corpus.", "intent": "Frecuencia muy baja"},
    {"id": 8, "text": "nonexistentword", "desc": "Consulta sin ningún término presente en el vocabulario global para probar robustez del algoritmo.", "intent": "Término fuera de vocabulario"}
]

print("=== Justificación de Consultas basada en DF ===")
for q in queries:
    tokens = re.findall(r'\b\w+\b', q['text'].lower())
    df_info = [f"{t} (DF: {df_global.get(t, 0)})" for t in tokens]
    print(f"Q{q['id']} - '{q['text']}': {', '.join(df_info)} -> {q['desc']}")

def get_tfidf_vector(tokens, idf_dict, vocab_index):
    vector = {}
    tf = Counter(tokens)
    for term, count in tf.items():
        if term in vocab_index:
            vector[term] = count * idf_dict[term]
    return vector

doc_tfidf_vectors = []
for tokens in corpus_tokens_list:
    doc_tfidf_vectors.append(get_tfidf_vector(tokens, idf_global, df_global))

def cosine_similarity_sparse(q_vector, d_vector):
    if not q_vector or not d_vector:
        return 0.0
    numerator = 0.0
    for term, q_val in q_vector.items():
        if term in d_vector:
            numerator += q_val * d_vector[term]

    q_norm = math.sqrt(sum(val**2 for val in q_vector.values()))
    d_norm = math.sqrt(sum(val**2 for val in d_vector.values()))

    if q_norm == 0 or d_norm == 0:
        return 0.0
    return numerator / (q_norm * d_norm)

avgdl = sum(len(d) for d in corpus_tokens_list) / len(corpus_tokens_list)

def compute_bm25(query_tokens, doc_tokens, idf_dict, k1=1.2, b=0.75, avgdl=avgdl):
    score = 0.0
    doc_len = len(doc_tokens)
    doc_tf = Counter(doc_tokens)

    for term in set(query_tokens):
        if term in idf_dict:
            tf = doc_tf.get(term, 0)
            if tf > 0:
                idf = idf_dict[term]
                numerator = idf * tf * (k1 + 1)
                denominator = tf + k1 * (1 - b + b * (doc_len / avgdl))
                score += numerator / denominator
    return score

=== Justificación de Consultas basada en DF ===
Q1 - 'alice in wonderland': alice (DF: 3), in (DF: 10), wonderland (DF: 2) -> Busca la obra de Alicia usando términos comunes en algunos libros pero nulos en otros.
Q2 - 'moby dick whale': moby (DF: 1), dick (DF: 1), whale (DF: 2) -> Busca el libro Moby Dick usando términos específicos de alta concentración en ese documento.
Q3 - 'the project gutenberg': the (DF: 10), project (DF: 10), gutenberg (DF: 10) -> Consulta con términos extremadamente comunes presentes en prácticamente todos los documentos.
Q4 - 'hunting of the snark': hunting (DF: 7), of (DF: 10), the (DF: 10), snark (DF: 1) -> Consulta de longitud media que coincide de manera muy exacta con el título de un documento específico (ID 13).
Q5 - 'world factbook central intelligence': world (DF: 9), factbook (DF: 1), central (DF: 3), intelligence (DF: 4) -> Consulta técnica con términos muy específicos del documento de datos (ID 14).
Q6 - 'bell captain sea': bell (DF: 5), captain (DF

**3. Compara los rankings**

Muestra por consulta una tabla como esta. Usa los IDs para comparar.

| Posición | ID y título TF-IDF | Coseno | ID y título BM25 | Score BM25 |
|---:|---|---:|---|---:|
| 1 | … | … | … | … |

Calcula cuántos IDs comparten los dos top 10 con $|T_{\mathrm{TFIDF}}\cap T_{\mathrm{BM25}}|$. Reporta también el tamaño real de cada lista y el primer documento de cada método. Dos listas pueden tener muchos IDs en común y ordenarlos de forma distinta.

Elige una consulta y dos documentos que cambien de posición o cuya distancia en el ranking cambie. Registra $|d|$, $f(t,d)$, $df(t)$ y el aporte BM25 de cada término de consulta para ambos documentos. Relaciona esas cifras con la saturación, la longitud o la rareza del término. Usa los scores de cada método para explicar su propio ranking.

In [23]:
from IPython.display import display, Markdown

all_results = []

for q in queries:
    q_text = q['text']
    q_tokens = re.findall(r'\b\w+\b', q_text.lower())

    q_tfidf_vec = get_tfidf_vector(q_tokens, idf_global, df_global)

    tfidf_scores = []
    bm25_scores = []

    for i, doc in enumerate(corpus):
        doc_id_str = str(doc['id'])
        doc_tokens = corpus_tokens_list[i]

        # TF-IDF score
        cos_sim = cosine_similarity_sparse(q_tfidf_vec, doc_tfidf_vectors[i])
        if cos_sim > 0:
            tfidf_scores.append((doc_id_str, doc['title'], cos_sim))

        # BM25 score
        bm_score = compute_bm25(q_tokens, doc_tokens, idf_global, k1=1.2, b=0.75)
        if bm_score > 0:
            bm25_scores.append((doc_id_str, doc['title'], bm_score))

    tfidf_scores = sorted(tfidf_scores, key=lambda x: (-x[2], x[0]))[:10]
    bm25_scores = sorted(bm25_scores, key=lambda x: (-x[2], x[0]))[:10]

    all_results.append({
        "query_id": q['id'],
        "query_text": q_text,
        "tfidf": tfidf_scores,
        "bm25": bm25_scores
    })

markdown_output = ""
for res in all_results:
    markdown_output += f"### Consulta {res['query_id']}: *'{res['query_text']}'*\n\n"
    markdown_output += "| Posición | ID y título TF-IDF | Coseno | ID y título BM25 | Score BM25 |\n"
    markdown_output += "|---:|---|---:|---|---:|\n"

    max_len = max(len(res['tfidf']), len(res['bm25']))
    if max_len == 0:
        markdown_output += "| - | No hay resultados | 0.0 | No hay resultados | 0.0 |\n"
    else:
        for idx in range(max_len):
            tf_str = f"{res['tfidf'][idx][0]} - {res['tfidf'][idx][1]}" if idx < len(res['tfidf']) else "-"
            tf_val = f"{res['tfidf'][idx][2]:.4f}" if idx < len(res['tfidf']) else "-"
            bm_str = f"{res['bm25'][idx][0]} - {res['bm25'][idx][1]}" if idx < len(res['bm25']) else "-"
            bm_val = f"{res['bm25'][idx][2]:.4f}" if idx < len(res['bm25']) else "-"
            markdown_output += f"| {idx+1} | {tf_str} | {tf_val} | {bm_str} | {bm_val} |\n"

    set_tfidf = set(x[0] for x in res['tfidf'])
    set_bm25 = set(x[0] for x in res['bm25'])
    intersection = set_tfidf.intersection(set_bm25)

    markdown_output += f"\n**Estadísticas de la consulta:**\n"
    markdown_output += f"- Tamaño real de la lista: TF-IDF = {len(res['tfidf'])} | BM25 = {len(res['bm25'])}.\n"
    markdown_output += f"- Cantidad de IDs compartidos en el top 10: {len(intersection)} ({list(intersection)}).\n\n"

display(Markdown(markdown_output))

### Consulta 1: *'alice in wonderland'*

| Posición | ID y título TF-IDF | Coseno | ID y título BM25 | Score BM25 |
|---:|---|---:|---|---:|
| 1 | 12 - Untitled | 0.5294 | 11 - Untitled | 2.4655 |
| 2 | 11 - Untitled | 0.5050 | 12 - Untitled | 2.1534 |
| 3 | 14 - Untitled | 0.0001 | 14 - Untitled | 0.3367 |

**Estadísticas de la consulta:**
- Tamaño real de la lista: TF-IDF = 3 | BM25 = 3.
- Cantidad de IDs compartidos en el top 10: 3 (['12', '11', '14']).

### Consulta 2: *'moby dick whale'*

| Posición | ID y título TF-IDF | Coseno | ID y título BM25 | Score BM25 |
|---:|---|---:|---|---:|
| 1 | 15 - Untitled | 0.3756 | 15 - Untitled | 5.8388 |
| 2 | 17 - Untitled | 0.0003 | 17 - Untitled | 0.6989 |

**Estadísticas de la consulta:**
- Tamaño real de la lista: TF-IDF = 2 | BM25 = 2.
- Cantidad de IDs compartidos en el top 10: 2 (['15', '17']).

### Consulta 3: *'the project gutenberg'*

| Posición | ID y título TF-IDF | Coseno | ID y título BM25 | Score BM25 |
|---:|---|---:|---|---:|
| - | No hay resultados | 0.0 | No hay resultados | 0.0 |

**Estadísticas de la consulta:**
- Tamaño real de la lista: TF-IDF = 0 | BM25 = 0.
- Cantidad de IDs compartidos en el top 10: 0 ([]).

### Consulta 4: *'hunting of the snark'*

| Posición | ID y título TF-IDF | Coseno | ID y título BM25 | Score BM25 |
|---:|---|---:|---|---:|
| 1 | 13 - Untitled | 0.5969 | 13 - Untitled | 2.4967 |
| 2 | 19 - Untitled | 0.0008 | 19 - Untitled | 0.3259 |
| 3 | 11 - Untitled | 0.0003 | 11 - Untitled | 0.2918 |
| 4 | 15 - Untitled | 0.0002 | 15 - Untitled | 0.2806 |
| 5 | 16 - Untitled | 0.0001 | 16 - Untitled | 0.2565 |
| 6 | 20 - Untitled | 0.0000 | 14 - Untitled | 0.2125 |
| 7 | 14 - Untitled | 0.0000 | 20 - Untitled | 0.1779 |

**Estadísticas de la consulta:**
- Tamaño real de la lista: TF-IDF = 7 | BM25 = 7.
- Cantidad de IDs compartidos en el top 10: 7 (['20', '19', '15', '16', '13', '14', '11']).

### Consulta 5: *'world factbook central intelligence'*

| Posición | ID y título TF-IDF | Coseno | ID y título BM25 | Score BM25 |
|---:|---|---:|---|---:|
| 1 | 14 - Untitled | 0.0086 | 14 - Untitled | 3.5143 |
| 2 | 15 - Untitled | 0.0030 | 15 - Untitled | 1.7769 |
| 3 | 18 - Untitled | 0.0020 | 18 - Untitled | 1.5525 |
| 4 | 20 - Untitled | 0.0003 | 20 - Untitled | 0.5566 |
| 5 | 17 - Untitled | 0.0001 | 17 - Untitled | 0.0993 |
| 6 | 11 - Untitled | 0.0000 | 11 - Untitled | 0.0939 |
| 7 | 12 - Untitled | 0.0000 | 16 - Untitled | 0.0930 |
| 8 | 16 - Untitled | 0.0000 | 12 - Untitled | 0.0910 |
| 9 | 19 - Untitled | 0.0000 | 19 - Untitled | 0.0907 |

**Estadísticas de la consulta:**
- Tamaño real de la lista: TF-IDF = 9 | BM25 = 9.
- Cantidad de IDs compartidos en el top 10: 9 (['20', '19', '15', '12', '16', '17', '14', '11', '18']).

### Consulta 6: *'bell captain sea'*

| Posición | ID y título TF-IDF | Coseno | ID y título BM25 | Score BM25 |
|---:|---|---:|---|---:|
| 1 | 15 - Untitled | 0.0551 | 16 - Untitled | 1.2975 |
| 2 | 13 - Untitled | 0.0388 | 13 - Untitled | 1.2373 |
| 3 | 16 - Untitled | 0.0298 | 15 - Untitled | 1.2136 |
| 4 | 12 - Untitled | 0.0038 | 12 - Untitled | 0.5987 |
| 5 | 17 - Untitled | 0.0011 | 14 - Untitled | 0.5607 |
| 6 | 14 - Untitled | 0.0002 | 17 - Untitled | 0.5436 |

**Estadísticas de la consulta:**
- Tamaño real de la lista: TF-IDF = 6 | BM25 = 6.
- Cantidad de IDs compartidos en el top 10: 6 (['15', '12', '16', '13', '17', '14']).

### Consulta 7: *'serpent dragon'*

| Posición | ID y título TF-IDF | Coseno | ID y título BM25 | Score BM25 |
|---:|---|---:|---|---:|
| 1 | 20 - Untitled | 0.0072 | 19 - Untitled | 0.8693 |
| 2 | 11 - Untitled | 0.0056 | 20 - Untitled | 0.8505 |
| 3 | 19 - Untitled | 0.0039 | 15 - Untitled | 0.6986 |
| 4 | 12 - Untitled | 0.0017 | 17 - Untitled | 0.5190 |
| 5 | 15 - Untitled | 0.0012 | 11 - Untitled | 0.4622 |
| 6 | 17 - Untitled | 0.0006 | 12 - Untitled | 0.4147 |
| 7 | 14 - Untitled | 0.0001 | 14 - Untitled | 0.4132 |

**Estadísticas de la consulta:**
- Tamaño real de la lista: TF-IDF = 7 | BM25 = 7.
- Cantidad de IDs compartidos en el top 10: 7 (['20', '19', '15', '12', '17', '14', '11']).

### Consulta 8: *'nonexistentword'*

| Posición | ID y título TF-IDF | Coseno | ID y título BM25 | Score BM25 |
|---:|---|---:|---|---:|
| - | No hay resultados | 0.0 | No hay resultados | 0.0 |

**Estadísticas de la consulta:**
- Tamaño real de la lista: TF-IDF = 0 | BM25 = 0.
- Cantidad de IDs compartidos en el top 10: 0 ([]).



**4. Identifica el efecto de los parámetros**

Mantén fijos la colección y el preprocesamiento. Escoge una consulta con coincidencias en documentos de distintas longitudes y frecuencias.

| Ejecución | $k_1$ | $b$ | Qué examinar |
|---|---:|---:|---|
| Base | 1.2 | 0.75 | Posiciones y aportes de referencia |
| Sin ajuste por longitud | 1.2 | 0.00 | Cambios en documentos más cortos o más largos que la media |
| Mayor efecto de las repeticiones | 2.0 | 0.75 | Cambios en el aporte de términos con frecuencias distintas |

Muestra las posiciones de los mismos documentos en las tres configuraciones. Si no se mueven, compara sus aportes por término y explica por qué el cambio de puntuación no alteró su orden.

In [24]:
configs = [
    {"name": "Base", "k1": 1.2, "b": 0.75},
    {"name": "Sin ajuste por longitud", "k1": 1.2, "b": 0.00},
    {"name": "Mayor efecto repeticiones", "k1": 2.0, "b": 0.75}
]

q_test_tokens = re.findall(r'\b\w+\b', "alice in wonderland".lower())

print("Efecto de los parámetros en el ranking de 'alice in wonderland' \n")
for cfg in configs:
    scores = []
    for i, doc in enumerate(corpus):
        score = compute_bm25(q_test_tokens, corpus_tokens_list[i], idf_global, k1=cfg['k1'], b=cfg['b'])
        if score > 0:
            scores.append((doc['id'], doc['title'], score))

    scores = sorted(scores, key=lambda x: (-x[2], x[0]))[:5]
    print(f"Configuración {cfg['name']} (k1={cfg['k1']}, b={cfg['b']}):")
    for rank, (d_id, title, val) in enumerate(scores, 1):
        print(f"  {rank}. ID {d_id} - Score: {val:.4f}")
    print()

Efecto de los parámetros en el ranking de 'alice in wonderland' 

Configuración Base (k1=1.2, b=0.75):
  1. ID 11 - Score: 2.4655
  2. ID 12 - Score: 2.1534
  3. ID 14 - Score: 0.3367

Configuración Sin ajuste por longitud (k1=1.2, b=0.0):
  1. ID 11 - Score: 2.2453
  2. ID 12 - Score: 1.8464
  3. ID 14 - Score: 0.5229

Configuración Mayor efecto repeticiones (k1=2.0, b=0.75):
  1. ID 11 - Score: 3.2037
  2. ID 12 - Score: 2.6780
  3. ID 14 - Score: 0.3121

